# Program 13: Prompt template over a loaded document

**Aim.** Load a text document (from Drive or disk) and create a prompt template that displays the output in a fixed layout.

**Theory.** A `PromptTemplate` has named slots. Fixing the output layout in the template makes answers consistent and easy to parse.

In [1]:
import sys
sys.path.insert(0, '.')
from common import *


In [2]:
# On Colab, mount Drive and point PATH at your file. Here the same step reads a file from disk.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PATH = '/content/drive/MyDrive/metformin.md'
except Exception:
    PATH = ROOT / 'data' / 'leaflets' / 'metformin.md'
doc = open(PATH, encoding='utf-8').read()
print(PATH, '-', len(doc.split()), 'words')

C:\Users\Sanjay S G\Documents\GMPE\data\leaflets\metformin.md - 348 words


In [3]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
template = PromptTemplate.from_template(
  'Read the leaflet and answer in EXACTLY this layout, using only the leaflet:\n'
  'MEDICINE: <name>\nUSED FOR: <one line>\nHOW TO TAKE: <one line>\nWARNINGS: <up to 3 short bullets>\n\nLEAFLET:\n{doc}')
print(template.format(doc='...')[:260], '\n')
llm, label = get_llm()
print('model:', label)
out = (template | llm | StrOutputParser()).invoke({'doc': doc})
print(out)
print('\nlayout kept:', all(h in out for h in ['MEDICINE:', 'USED FOR:', 'HOW TO TAKE:', 'WARNINGS:']))

Read the leaflet and answer in EXACTLY this layout, using only the leaflet:
MEDICINE: <name>
USED FOR: <one line>
HOW TO TAKE: <one line>
WARNINGS: <up to 3 short bullets>

LEAFLET:
... 


model: Groq openai/gpt-oss-20b (no Cohere key available)


MEDICINE: Metformin tablets  
USED FOR: Lowers blood sugar in type 2 diabetes  
HOW TO TAKE: Take with or just after meals, swallow whole with water, at the same time each day  
WARNINGS:  
- Kidney, liver, heart problems or heavy alcohol use  
- Stop before contrast‑enhanced scans or surgery  
- Lactic acidosis: vomiting, fast breathing, muscle cramps, weakness – seek urgent help

layout kept: True


## Observations

A fixed template plus a check on the headings lets code rely on the output. Content accuracy still needs checking against the leaflet.